# U-Net 학습: 마스크 준비 → Training Job 제출

**전체 흐름** (위에서부터 순서대로 실행하고 자면 됨):
1. CheXmask CSV 다운로드 (4.4GB, ~10분)
2. 전체 MIMIC-CXR 마스크 디코딩 + NPZ 저장 (~30분)
3. Manifest CSV 생성 + S3 업로드
4. train_unet.py 패키징 + S3 업로드
5. SageMaker Training Job 제출 (스팟, ~2-4시간)

> Training Job은 제출 후 독립 실행됨. 노트북 꺼도 됨.

## Part 1: 환경 설정

In [ ]:
import os, time, json, tarfile, subprocess
import numpy as np
import pandas as pd
from PIL import Image
import boto3

WORK_DIR = '/home/ec2-user/SageMaker/unet_data'
os.makedirs(WORK_DIR, exist_ok=True)

BUCKET = 'pre-project-practice-hyunwoo-666803869796-ap-northeast-2-an'
IMAGE_BUCKET = 'say1-pre-project-5'  # 공용 CXR 이미지 버킷
S3_MASKS_PREFIX = 'data/unet_masks'
S3_MANIFEST_PREFIX = 'data/unet_manifest'
S3_CODE_PREFIX = 'code'

CHEXMASK_CSV = os.path.join(WORK_DIR, 'MIMIC-CXR-JPG.csv')
SPLIT_CSV = os.path.join(WORK_DIR, 'mimic-cxr-2.0.0-split.csv')
OUTPUT_DIR = os.path.join(WORK_DIR, 'masks_512')
TARGET_SIZE = 512

sm_client = boto3.client('sagemaker', region_name='ap-northeast-2')
s3_client = boto3.client('s3')

print(f"작업 버킷: {BUCKET}")
print(f"이미지 버킷: {IMAGE_BUCKET}")
print("환경 설정 완료")

## Part 2: CheXmask 다운로드 + Split CSV 준비

In [ ]:
%%time
# CheXmask Preprocessed CSV 다운로드 (4.4GB)
CHEXMASK_URL = "https://physionet.org/files/chexmask-cxr-segmentation-data/1.0.0/Preprocessed/MIMIC-CXR-JPG.csv"

if os.path.exists(CHEXMASK_CSV):
    print(f"이미 존재: {os.path.getsize(CHEXMASK_CSV)/(1024**3):.2f} GB")
else:
    print("CheXmask CSV 다운로드 (~4.4 GB, ~10분)...")
    !wget -O {CHEXMASK_CSV} --progress=bar:force "{CHEXMASK_URL}"
    print(f"완료: {os.path.getsize(CHEXMASK_CSV)/(1024**3):.2f} GB")

# Split CSV
if not os.path.exists(SPLIT_CSV):
    !aws s3 cp s3://{BUCKET}/data/mimic-cxr-csv/mimic-cxr-2.0.0-split.csv {SPLIT_CSV}
print(f"Split CSV: {SPLIT_CSV}")

## Part 3: RLE 디코딩 함수 + 전체 CSV 로드/필터

In [ ]:
def rle_to_mask(rle_string, height, width):
    if pd.isna(rle_string) or str(rle_string).strip() in ('', 'nan'):
        return np.zeros((height, width), dtype=np.uint8)
    runs = np.array([int(x) for x in str(rle_string).split()])
    starts, lengths = runs[0::2], runs[1::2]
    mask = np.zeros(height * width, dtype=np.uint8)
    for s, l in zip(starts, lengths):
        mask[s-1:s-1+l] = 1
    return mask.reshape((height, width))

def decode_masks(row, target_size=512):
    h, w = int(row['Height']), int(row['Width'])
    ll = rle_to_mask(row.get('Left Lung', ''), h, w)
    rl = rle_to_mask(row.get('Right Lung', ''), h, w)
    ht = rle_to_mask(row.get('Heart', ''), h, w)
    if h != target_size or w != target_size:
        ll = np.array(Image.fromarray(ll).resize((target_size, target_size), Image.NEAREST))
        rl = np.array(Image.fromarray(rl).resize((target_size, target_size), Image.NEAREST))
        ht = np.array(Image.fromarray(ht).resize((target_size, target_size), Image.NEAREST))
    combined = np.zeros((target_size, target_size), dtype=np.uint8)
    combined[ll > 0] = 1
    combined[rl > 0] = 2
    combined[ht > 0] = 3
    return combined

print("함수 정의 완료")

In [ ]:
%%time
# 전체 CSV 로드 + 품질 필터 (p10 필터 없음 — 전체 MIMIC-CXR)
print("전체 CSV 로드 중 (4.4 GB)... 2~5분")
df = pd.read_csv(CHEXMASK_CSV)
print(f"전체: {len(df):,}행")

# 품질 필터
dice_col = [c for c in df.columns if 'dice' in c.lower() and 'mean' in c.lower()]
if dice_col:
    dice_col = dice_col[0]
    before = len(df)
    df = df[df[dice_col] >= 0.7].copy()
    print(f"품질 필터 ({dice_col} >= 0.7): {before:,} → {len(df):,}")

# ImageID 컬럼 + dicom_id 추출
id_col = df.columns[0]
df['dicom_id'] = df[id_col].apply(lambda x: os.path.splitext(os.path.basename(str(x)))[0])
print(f"ImageID 샘플: {df[id_col].iloc[0]}")
print(f"dicom_id 샘플: {df['dicom_id'].iloc[0]}")

# Split 병합
split_df = pd.read_csv(SPLIT_CSV)
df = df.merge(split_df[['dicom_id', 'split']], on='dicom_id', how='inner')
print(f"\nSplit 병합 후: {len(df):,}행")
print(df['split'].value_counts())

## Part 4: 전체 마스크 디코딩 → NPZ 저장 (~30분)

In [ ]:
%%time

for split_name in ['train', 'validate', 'test']:
    split_df = df[df['split'] == split_name]
    if len(split_df) == 0:
        print(f"[{split_name}] 없음"); continue

    out_dir = os.path.join(OUTPUT_DIR, split_name)
    os.makedirs(out_dir, exist_ok=True)

    # 이미 처리된 파일 스킵
    existing = set(f.replace('.npz','') for f in os.listdir(out_dir) if f.endswith('.npz'))
    todo_df = split_df[~split_df['dicom_id'].isin(existing)]

    if len(todo_df) == 0:
        print(f"[{split_name}] 이미 완료 ({len(existing):,}개)")
        continue

    total = len(todo_df)
    saved, errors = 0, 0
    print(f"\n{'='*50}")
    print(f"[{split_name}] {total:,}개 처리 (기존 {len(existing):,}개 스킵)")

    for idx, (_, row) in enumerate(todo_df.iterrows()):
        try:
            combined = decode_masks(row, TARGET_SIZE)
            np.savez_compressed(
                os.path.join(out_dir, f"{row['dicom_id']}.npz"),
                combined=combined, image_id=str(row[id_col])
            )
            saved += 1
        except Exception as e:
            errors += 1
            if errors <= 3: print(f"  오류: {e}")

        if (idx+1) % 2000 == 0 or idx == total-1:
            pct = (idx+1)/total*100
            print(f"  {idx+1:,}/{total:,} ({pct:.1f}%) | 저장:{saved:,} 오류:{errors}")

    print(f"[{split_name}] 완료: {saved:,}개")

print("\n전체 디코딩 완료!")

## Part 5: Manifest CSV 생성 + S3 업로드

Training Job이 이미지-마스크를 매칭하기 위한 manifest.
`image_path`는 say1-pre-project-5 내의 상대경로 = CheXmask의 ImageID.

In [ ]:
# Manifest CSV: dicom_id, image_path (S3 key in say1-pre-project-5), split
manifest = df[[id_col, 'dicom_id', 'split']].copy()
manifest.rename(columns={id_col: 'image_path'}, inplace=True)

# 실제 존재하는 NPZ만 포함
valid_ids = set()
for split_name in ['train', 'validate', 'test']:
    d = os.path.join(OUTPUT_DIR, split_name)
    if os.path.exists(d):
        valid_ids.update(f.replace('.npz','') for f in os.listdir(d) if f.endswith('.npz'))

before = len(manifest)
manifest = manifest[manifest['dicom_id'].isin(valid_ids)]
print(f"Manifest: {before:,} → {len(manifest):,} (NPZ 존재하는 것만)")
print(manifest['split'].value_counts())

# 로컬 저장
manifest_path = os.path.join(WORK_DIR, 'unet_manifest.csv')
manifest.to_csv(manifest_path, index=False)
print(f"\n저장: {manifest_path}")
print(manifest.head())

In [ ]:
%%time
# S3 업로드: 마스크 NPZ + Manifest CSV
print("마스크 NPZ S3 업로드...")
!aws s3 sync {OUTPUT_DIR} s3://{BUCKET}/{S3_MASKS_PREFIX}/ --quiet

print("Manifest CSV S3 업로드...")
!aws s3 cp {manifest_path} s3://{BUCKET}/{S3_MANIFEST_PREFIX}/unet_manifest.csv

print("\n업로드 완료!")
!aws s3 ls s3://{BUCKET}/{S3_MASKS_PREFIX}/ 
!aws s3 ls s3://{BUCKET}/{S3_MANIFEST_PREFIX}/

## Part 6: 학습 코드 패키징 + S3 업로드

`train_unet.py`를 `sourcedir.tar.gz`로 압축하여 S3에 업로드.
SageMaker Training Job이 이 파일을 다운로드하여 실행.

In [ ]:
# train_unet.py를 tar.gz로 패키징
# 프로젝트 코드가 SageMaker 노트북에 있다면 그 경로 사용
# 없으면 S3에서 다운로드

TRAIN_SCRIPT = None

# 경로 후보 (SageMaker 노트북에서 git clone 했을 수 있음)
candidates = [
    '/home/ec2-user/SageMaker/forpreproject/layer1_segmentation/train_unet.py',
    os.path.join(WORK_DIR, 'train_unet.py'),
]

for path in candidates:
    if os.path.exists(path):
        TRAIN_SCRIPT = path
        break

if TRAIN_SCRIPT is None:
    # S3에서 다운로드 시도
    print("train_unet.py를 S3에서 다운로드...")
    TRAIN_SCRIPT = os.path.join(WORK_DIR, 'train_unet.py')
    !aws s3 cp s3://{BUCKET}/code/train_unet.py {TRAIN_SCRIPT} 2>/dev/null || true
    
    if not os.path.exists(TRAIN_SCRIPT):
        print("ERROR: train_unet.py를 찾을 수 없습니다!")
        print("로컬에서 먼저 업로드하세요:")
        print(f"  aws s3 cp layer1_segmentation/train_unet.py s3://{BUCKET}/code/train_unet.py")
        raise FileNotFoundError("train_unet.py")

print(f"학습 스크립트: {TRAIN_SCRIPT}")

# tar.gz 생성
tarball_path = os.path.join(WORK_DIR, 'unet_sourcedir.tar.gz')
with tarfile.open(tarball_path, 'w:gz') as tar:
    tar.add(TRAIN_SCRIPT, arcname='train_unet.py')
print(f"패키징 완료: {tarball_path} ({os.path.getsize(tarball_path)/1024:.1f} KB)")

# S3 업로드
s3_code_key = f'{S3_CODE_PREFIX}/unet_sourcedir.tar.gz'
s3_client.upload_file(tarball_path, BUCKET, s3_code_key)
print(f"업로드: s3://{BUCKET}/{s3_code_key}")

## Part 7: SageMaker Training Job 제출

**이 셀 실행 후 자도 됩니다!**

스팟 인스턴스 사용 (ml.g5.xlarge). 예상 비용: ~$2-4 (2-4시간).
Job 상태는 내일 아침에 확인:
```python
sm_client.describe_training_job(TrainingJobName='unet-lung-heart-v1')['TrainingJobStatus']
```

In [ ]:
JOB_NAME = 'unet-lung-heart-v1'

training_params = {
    "TrainingJobName": JOB_NAME,
    "RoleArn": "arn:aws:iam::666803869796:role/SKKU_SageMaker_Role",
    "AlgorithmSpecification": {
        "TrainingImage": "763104351884.dkr.ecr.ap-northeast-2.amazonaws.com/pytorch-training:2.8.0-gpu-py312-cu129-ubuntu22.04-sagemaker",
        "TrainingInputMode": "File"
    },
    "HyperParameters": {
        "sagemaker_program": "train_unet.py",
        "sagemaker_submit_directory": f"s3://{BUCKET}/{s3_code_key}",
        "sagemaker_region": "ap-northeast-2",
        "epochs": "50",
        "batch-size": "8",
        "lr": "0.0001",
        "image-size": "512",
        "num-workers": "4",
        "image-bucket": IMAGE_BUCKET   # S3 직접 다운로드 (채널 대신)
    },
    "ResourceConfig": {
        "InstanceType": "ml.g5.xlarge",
        "InstanceCount": 1,
        "VolumeSizeInGB": 150
    },
    "InputDataConfig": [
        {
            "ChannelName": "masks",
            "DataSource": {
                "S3DataSource": {
                    "S3DataType": "S3Prefix",
                    "S3Uri": f"s3://{BUCKET}/{S3_MASKS_PREFIX}/",
                    "S3DataDistributionType": "FullyReplicated"
                }
            }
        },
        {
            "ChannelName": "manifest",
            "DataSource": {
                "S3DataSource": {
                    "S3DataType": "S3Prefix",
                    "S3Uri": f"s3://{BUCKET}/{S3_MANIFEST_PREFIX}/",
                    "S3DataDistributionType": "FullyReplicated"
                }
            }
        }
    ],
    "CheckpointConfig": {
        "S3Uri": f"s3://{BUCKET}/checkpoints/unet-lung-heart/"
    },
    "OutputDataConfig": {
        "S3OutputPath": f"s3://{BUCKET}/output/"
    },
    "EnableManagedSpotTraining": True,
    "StoppingCondition": {
        "MaxRuntimeInSeconds": 28800,
        "MaxWaitTimeInSeconds": 172800
    },
    "Tags": [
        {"Key": "name", "Value": "say2-preproject-6team-hyunwoo"}
    ]
}

print("=" * 60)
print(f"Training Job 제출: {JOB_NAME}")
print(f"인스턴스: ml.g5.xlarge (GPU A10G 24GB)")
print(f"이미지: S3 직접 다운로드 (s3://{IMAGE_BUCKET}/)")
print(f"  → images 채널 제거, 필요한 이미지만 선택 다운로드")
print(f"스팟 학습: 활성화 (비용 ~70% 절감)")
print(f"예상 소요: 2~4시간 (이미지 다운로드 ~20분 + 학습 ~2-3시간)")
print(f"예상 비용: ~$2-4")
print("=" * 60)

response = sm_client.create_training_job(**training_params)
print(f"\n제출 완료! ARN: {response['TrainingJobArn']}")
print(f"\n내일 아침에 확인:")
print(f"  sm_client.describe_training_job(TrainingJobName='{JOB_NAME}')['TrainingJobStatus']")

## Part 8: (선택) 상태 확인

아래는 내일 아침에 실행하여 결과 확인하는 셀.

In [ ]:
# Training Job 상태 확인
JOB_NAME = 'unet-lung-heart-v1'
desc = sm_client.describe_training_job(TrainingJobName=JOB_NAME)

print(f"Job: {JOB_NAME}")
print(f"Status: {desc['TrainingJobStatus']}")
if 'SecondaryStatus' in desc:
    print(f"Detail: {desc['SecondaryStatus']}")
if desc['TrainingJobStatus'] == 'Completed':
    metrics = desc.get('FinalMetricDataList', [])
    print(f"\nTraining Time: {desc.get('TrainingTimeInSeconds', 0)/60:.1f}분")
    print(f"Billable Time: {desc.get('BillableTimeInSeconds', 0)/60:.1f}분")
    print(f"Model: s3://{BUCKET}/output/{JOB_NAME}/output/model.tar.gz")
elif desc['TrainingJobStatus'] == 'Failed':
    print(f"\nFailure Reason: {desc.get('FailureReason', 'unknown')}")

In [ ]:
# 학습 완료 후 결과 다운로드
JOB_NAME = 'unet-lung-heart-v1'
MODEL_TAR = f'/tmp/{JOB_NAME}_model.tar.gz'

!aws s3 cp s3://{BUCKET}/output/{JOB_NAME}/output/model.tar.gz {MODEL_TAR}

import tarfile
with tarfile.open(MODEL_TAR, 'r:gz') as tar:
    tar.extractall('/tmp/unet_model/')

# 결과 확인
with open('/tmp/unet_model/results.json') as f:
    results = json.load(f)

print("=" * 50)
print("U-Net 학습 결과")
print("=" * 50)
print(f"Mean Dice: {results['mean_dice']:.4f}")
print(f"  Left Lung:  {results['per_class_dice']['left_lung']:.4f}")
print(f"  Right Lung: {results['per_class_dice']['right_lung']:.4f}")
print(f"  Heart:      {results['per_class_dice']['heart']:.4f}")
print(f"학습 시간: {results['training_time_minutes']:.1f}분")
print(f"Best Val Dice: {results['best_val_dice']:.4f}")

# 모델을 Lambda 배포용 S3 경로에 복사
!aws s3 cp /tmp/unet_model/best_model.pth s3://{BUCKET}/models/unet_lung_heart.pth
!aws s3 cp /tmp/unet_model/model_info.json s3://{BUCKET}/models/unet_model_info.json
print(f"\nLambda 배포용 모델 업로드 완료:")
print(f"  s3://{BUCKET}/models/unet_lung_heart.pth")
print(f"  s3://{BUCKET}/models/unet_model_info.json")